# Mixl1 Cell Calling
#### Mai-Linh Ton, 18/05/2020

In [2]:
library(DropletUtils)
library(ggplot2)
library(Matrix)
library(BiocParallel)
ncores = 3
mcparam = MulticoreParam(workers = ncores)
register(mcparam)

library(knitr)
library(reshape2)

Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: parallel


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:parallel’:

    clusterApply, clusterApplyLB, clusterCall, clusterEvalQ,
    clusterExport, clusterMap, parApply, parCapply, parLapply,
    parLapplyLB, parRapply, parSapply, parSapplyLB


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tap

In [3]:
##===Read in files===##
wd <- paste0("/rds/project/bg200/rds-bg200-hphi-gottgens/users/mlnt2/PhD_MT07_Mixl1/1_indexCorrection/SIGA",LETTERS[1:6],"5/")

out_loc <- paste0(wd, "matrix.mtx")
bc_loc <- paste0(wd, "barcodes.tsv")
gene_loc <- paste0(wd, "genes.tsv")

In [4]:
# read matrices and tables into a single variable
matrices = bplapply(out_loc, readMM)
bcs = bplapply(bc_loc, function(x) read.table(x, header = FALSE, stringsAsFactors = FALSE)[,1])
               
                    
#correct barcode sample number
for(i in 1:length(bcs)){
  bcs[[i]] = paste0(bcs[[i]], "-", i)
}

In [5]:
##==Create a dataframe with the sample metadata/parameters==##

sample = 1:6
stage = rep("8.5", 6)
batch = rep("1", 6)
embryo_pool = 1:6
ID = paste0("SIGA",LETTERS[1:6],"5")

exp_design <- data.frame(sample, stage, batch, embryo_pool, ID)
exp_design

sample,stage,batch,embryo_pool,ID
<int>,<fct>,<fct>,<int>,<fct>
1,8.5,1,1,SIGAA5
2,8.5,1,2,SIGAB5
3,8.5,1,3,SIGAC5
4,8.5,1,4,SIGAD5
5,8.5,1,5,SIGAE5
6,8.5,1,6,SIGAF5


In [6]:
##==Do cell calling==##

set.seed(42)
#do call
outs = lapply(matrices, emptyDrops, niters = 20000, ignore = 4999, BPPARAM = mcparam, lower = 100, retain = Inf)


#identify cells
sigs = lapply(outs, function(x) x$FDR <= 0.01 & !is.na(x$FDR))

#subset the cells
cells = lapply(1:length(matrices), function(i) matrices[[i]][, sigs[[i]]])
barcodes = lapply(1:length(bcs), function(i) bcs[[i]][sigs[[i]]])

#append
counts = do.call(cbind, cells)
barcodes = do.call(c, barcodes)

In [7]:
##==save==##
out_dir <- ("/rds/project/bg200/rds-bg200-hphi-gottgens/users/mlnt2/PhD_MT07_Mixl1/2_calledCells/")
 
writeMM(counts, file = paste0(out_dir,"raw_counts.mtx"))
write.table(barcodes, file = paste0(out_dir, "barcodes.tsv"), col.names = FALSE, row.names = FALSE, quote = FALSE)
file.copy(from = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/mlnt2/PhD_MT07_Mixl1/1_indexCorrection/SIGAA5/genes.tsv",
          to = paste0(out_dir, "genes.tsv"), overwrite = TRUE)

NULL

[1] TRUE

In [8]:
sessionInfo()

R version 3.6.1 (2019-07-05)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Scientific Linux 7.7 (Nitrogen)

Matrix products: default
BLAS:   /usr/local/software/spack/spack-0.11.2/opt/spack/linux-rhel7-x86_64/gcc-5.4.0/r-3.6.1-zrytncqvsnw5h4dl6t6njefj7otl4bg4/rlib/R/lib/libRblas.so
LAPACK: /usr/local/software/spack/spack-0.11.2/opt/spack/linux-rhel7-x86_64/gcc-5.4.0/r-3.6.1-zrytncqvsnw5h4dl6t6njefj7otl4bg4/rlib/R/lib/libRlapack.so

locale:
 [1] LC_CTYPE=en_GB.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_GB.UTF-8        LC_COLLATE=en_GB.UTF-8    
 [5] LC_MONETARY=en_GB.UTF-8    LC_MESSAGES=en_GB.UTF-8   
 [7] LC_PAPER=en_GB.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_GB.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] parallel  stats4    stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] reshape2_1.4.3              knitr_1.28  